# Math 638 - Homework 7.1

## Conditioned Generation: From Parameters to Dynamics to Images

This report summarizes the completed Homework 7.1 project.

The scientific pipeline is

$$
c \longmapsto \Gamma_c \longmapsto I_c
$$

where:

- $c$ is the condition vector;
- $\Gamma_c$ is the sampled Lorenz trajectory;
- $I_c$ is the rendered image.

The image is not the primary mathematical object. The condition first determines a dynamical system and its trajectory. The trajectory can then be checked mathematically before it is rendered.

## Executive Summary

The condition vector is

$$
c = (\sigma,\rho,\beta,x_0,y_0,z_0,T).
$$

For the reference simulation,

$$
\sigma = 10,
$$

$$
\rho = 28,
$$

$$
\beta = \frac{8}{3},
$$

$$
(x_0,y_0,z_0) = (1,1,1),
$$

and

$$
T = 25.
$$

The trajectory is sampled with

$$
\Delta t = 0.01.
$$

This gives 2501 sampled states and a flattened trajectory dimension of

$$
7503.
$$

The rendered image has size

$$
256 \times 256 \times 3,
$$

so its flattened dimension is

$$
196608.
$$

The reference numerical trajectory is consistent with the Lorenz differential equations at the chosen sampling scale. The final relative vector RMSE is approximately

$$
0.004948.
$$

The conditioned dataset contains 120 samples: 84 training, 18 validation, and 18 test conditions.

The final scientific conclusion favors generating a mathematical structure first and rendering it afterward when a governing structure is known.

## Deliverables Overview

| Part | Deliverable | Status |
| --- | --- | --- |
| I | Define condition, trajectory, and image spaces | Complete |
| II | Simulate Lorenz dynamics and verify the ODE numerically | Complete |
| III | Render trajectories as images and analyze information loss | Complete |
| IV | Build a conditioned dataset | Complete |
| V | Reflect using compiled numerical evidence | Complete |

# Part I - Define the Spaces

## Condition space

The condition vector is

$$
c = (\sigma,\rho,\beta,x_0,y_0,z_0,T).
$$

The Lorenz system is

$$
\dot{x} = \sigma(y-x),
$$

$$
\dot{y} = x(\rho-z)-y,
$$

$$
\dot{z} = xy-\beta z.
$$

The dataset parameter ranges are

$$
8 \le \sigma \le 12,
$$

$$
24 \le \rho \le 32,
$$

$$
2.2 \le \beta \le 3.0,
$$

$$
-15 \le x_0 \le 15,
$$

$$
-15 \le y_0 \le 15,
$$

$$
5 \le z_0 \le 35,
$$

with

$$
T = 25.
$$

## Trajectory and image spaces

For a fixed condition $c$, the continuous trajectory is

$$
\gamma_c(t) = (x(t),y(t),z(t)).
$$

The stored sampled trajectory is

$$
\Gamma_c =
(x(t_0),y(t_0),z(t_0),\ldots,x(t_M),y(t_M),z(t_M)).
$$

With $T=25$ and $\Delta t=0.01$,

$$
M = 2500
$$

and

$$
M+1 = 2501.
$$

The flattened trajectory dimension is

$$
3(2501) = 7503.
$$

Each rendered image has 256 by 256 pixels and three RGB channels, so the flattened image dimension is

$$
3(256)(256) = 196608.
$$

The dimension chain is

$$
R^7 \longmapsto R^{7503} \longmapsto R^{196608}.
$$

The larger image dimension does not imply greater scientific information.

# Part II - Numerical Simulation

The reference condition is

$$
c_* = (10,28,\frac{8}{3},1,1,1,25).
$$

The Lorenz system is integrated with DOP853 using tight numerical tolerances.

The stored trajectory contains 2501 samples.

## Time series

![Lorenz time series](../result%20images/hw7_1/part2_lorenz_time_series.png)

## Three-dimensional phase portrait

![Lorenz 3D phase portrait](../result%20images/hw7_1/part2_lorenz_phase_portrait_3d.png)

## Two-dimensional projection

![Lorenz x-z projection](../result%20images/hw7_1/part2_lorenz_xz_projection.png)

## Numerical Verification of the Lorenz Equations

For interior samples, the centered finite-difference approximation is

$$
\dot{\Gamma}(t_i)
\approx
\frac{\Gamma(t_{i+1})-\Gamma(t_{i-1})}{2\Delta t}.
$$

The first and last samples are excluded from this centered-difference summary.

| Component | Residual RMSE | RHS RMS | Relative RMSE | Maximum absolute residual | Median absolute residual |
| --- | ---: | ---: | ---: | ---: | ---: |
| x | 0.121915 | 35.980007 | 0.003388 | 1.009057 | 0.027285 |
| y | 0.315924 | 55.534392 | 0.005689 | 3.817776 | 0.041525 |
| z | 0.317614 | 66.525989 | 0.004774 | 3.496838 | 0.046002 |

The relative vector RMSE is

$$
0.004948.
$$

These values support numerical consistency with the Lorenz equations at the chosen sampling scale. They do not prove exact integration or arbitrarily long shadowing of one exact chaotic trajectory.

![Lorenz ODE residual](../result%20images/hw7_1/part2_ode_residual_norm.png)

# Part III - From Trajectories to Images

The rendering uses the $x$-$z$ projection.

Physical coordinates are mapped to a fixed 256-by-256 raster. Consecutive projected states are joined by line segments and stored as an RGB image.

The image shape is

$$
256 \times 256 \times 3.
$$

The flattened image dimension is

$$
196608.
$$

The normalized pixel range is from 0 to 1.

![Rendered trajectory image](../result%20images/hw7_1/part3_reference_rendered_image_display.png)

## Why the rendering map is not one-to-one

The rendering map is many-to-one.

The renderer keeps only the $x$ and $z$ coordinates, so the full three-dimensional state cannot be reconstructed.

Continuous coordinates are also mapped to a finite pixel grid. Different physical coordinates can occupy the same pixel.

The phase portrait does not encode the exact sample time associated with every point, and it does not directly encode speed along the path.

Rasterization introduces additional information loss because a finite-width line represents a region rather than one exact mathematical coordinate.

The image is therefore a useful visualization, but it is not an invertible representation of the numerical trajectory.

# Part IV - Conditioned Dataset

The project constructs 120 independently conditioned Lorenz simulations.

| Split | Samples |
| --- | ---: |
| Training | 84 |
| Validation | 18 |
| Test | 18 |
| Total | 120 |

Every sample stores:

1. one seven-coordinate condition vector;
2. one trajectory with shape 2501 by 3;
3. one 256-by-256 RGB image.

The dataset stores the three objects as corresponding records. This pairing does not mean that the maps are invertible.

## Rendering protocol and verification

All dataset images use one common rendering map.

The physical rendering bounds are estimated from training trajectories only. A 5 percent padding margin is added, and those bounds are frozen before validation and test images are rendered.

For the final generated dataset:

- mean validation fraction outside the frozen bounds: 0;
- mean test fraction outside the frozen bounds: 0;
- maximum validation fraction outside the frozen bounds: 0;
- maximum test fraction outside the frozen bounds: 0.

Thus no validation or test trajectory points were clipped by the training-fitted rendering range.

All 120 samples passed the stored condition, trajectory, and image shape checks.

Selected saved images were also independently regenerated from the saved numerical trajectories and matched pixel-for-pixel.

## Dataset preview

The preview uses fixed sample IDs rather than selecting examples by visual quality.

![Conditioned dataset preview](../result%20images/hw7_1/part4_conditioned_dataset_preview.png)

The scientific generation chain remains explicit:

$$
c \longmapsto \Gamma_c \longmapsto I_c.
$$

# Part V - Reflection

## Why is this a cleaner scientific analogue of conditioned image generation?

The Lorenz experiment is a cleaner scientific analogue because every stage has a clear mathematical meaning.

The condition vector specifies the differential equation, initial state, and integration horizon.

The numerical trajectory is not accepted only because it looks like a Lorenz attractor. Part II checks the trajectory against the Lorenz equations using finite-difference derivative estimates.

The relative component RMSE values are approximately

$$
0.00339,
$$

$$
0.00569,
$$

and

$$
0.00477.
$$

The relative vector RMSE is approximately

$$
0.00495.
$$

These values give quantitative evidence that the stored trajectory is consistent with the governing equations at the selected numerical resolution.

Part III shows that the rendered image is a lossy representation. It omits the $y$ coordinate, exact time information, speed information, and exact numerical state values.

A visually plausible image alone therefore cannot certify that the underlying dynamics satisfy the Lorenz equations.

Part IV extends the same idea to 120 independently conditioned examples. Every condition remains paired with both the numerical trajectory and the rendered image.

This separates what was conditioned on, what mathematical object was generated, and how that object was visualized. That separation makes scientific correctness testable rather than purely visual.

## Should scientific generative AI map conditions directly to images?

For a problem with a known governing mathematical structure, the results support a structure-first pipeline.

A structure-first model has the form

$$
c \longmapsto \widehat{\Gamma}_c \longmapsto I_c.
$$

A direct image-only model has the form

$$
c \longmapsto I.
$$

A direct image model could produce a visually convincing Lorenz-like picture without producing any state sequence that satisfies the Lorenz equations.

A structure-first model exposes an intermediate mathematical object that can be tested before rendering.

For example, a predicted trajectory can be checked against governing equations, physical constraints, or long-run statistics.

This does not mean that structure-first generation is automatically correct.

If a learned model predicts

$$
\widehat{\Gamma}_c,
$$

the predicted trajectory still requires validation. It could violate the equations, respond incorrectly to parameter changes, or have incorrect long-run behavior.

The advantage is that these failures remain measurable.

Direct image generation can still be reasonable when the image itself is the scientific observable or when no reliable intermediate mathematical structure is known.

When governing structure is known and scientific validity matters, the evidence from this project favors generating the mathematical structure before rendering the final image.

# Reproducibility and Qualifications

The final folder contains five executed notebooks:

- `HW7_1_PartI_Define_the_Spaces.ipynb`
- `HW7_1_PartII_Numerical_Simulation.ipynb`
- `HW7_1_PartIII_From_Trajectories_to_Images.ipynb`
- `HW7_1_PartIV_Build_Conditioned_Dataset.ipynb`
- `HW7_1_PartV_Reflection_After_Compilation.ipynb`

The saved outputs include the reference condition, reference trajectory, ODE residual diagnostics, rendering configuration, 120-condition dataset manifest, numerical trajectories, rendered images, and dataset verification results.

Important qualifications remain:

- a small local residual does not prove arbitrarily long exact shadowing in a chaotic system;
- the sampled parameter range includes different Lorenz regimes;
- the project constructs a simulator and conditioned dataset, not a trained neural condition-to-image generator;
- a higher-dimensional image vector can still be a lossy representation.

# Overall Conclusion

Homework 7.1 demonstrates a scientific conditioned-generation pipeline in which the mathematical structure remains explicit.

The completed pipeline is

$$
c \longmapsto \Gamma_c \longmapsto I_c.
$$

The numerical evidence supports three main conclusions.

First, the reference trajectory is quantitatively consistent with the Lorenz differential equations at the chosen sampling resolution.

Second, the trajectory-to-image rendering is many-to-one and removes information needed for full scientific verification.

Third, when a known governing mathematical structure exists, generating or predicting that structure before rendering provides a clearer path to validation than generating only a plausible-looking image.

The final image is best understood as a visualization of a scientific object, not as a replacement for the scientific object itself.